# SVM Implementation Assignment

## Objective
Implement Support Vector Machine (SVM) classifiers with different kernel variations (Linear, Polynomial, RBF, and Sigmoid) from scratch and compare their performance on the Portuguese Bank Market Dataset.

In [1]:
# Only allowed libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

## SVM Class Implementation
Here we implement the SVM class using the Simplified Sequential Minimal Optimization (SMO) algorithm.

In [2]:
import numpy as np

class SVM:
    def __init__(self, kernel='linear', C=1.0, degree=3, gamma=0.1, tol=1e-3, max_iter=100):
        self.kernel_type = kernel
        self.C = C
        self.degree = degree
        self.gamma = gamma
        self.tol = tol
        self.max_iter = max_iter
        self.alpha = None
        self.b = 0
        self.X = None
        self.y = None
        self.support_vectors = None
        self.support_vector_labels = None
        self.support_vector_alphas = None

    def _kernel(self, x1, x2):
        if self.kernel_type == 'linear':
            return np.dot(x1, x2.T)
        elif self.kernel_type == 'poly':
            return (np.dot(x1, x2.T) + 1) ** self.degree
        elif self.kernel_type == 'rbf':
            # RBF kernel: exp(-gamma * ||x1 - x2||^2)
            # Utilizing the expansion: ||x - y||^2 = ||x||^2 + ||y||^2 - 2 <x, y>
            # This is complex to vectorize efficiently for all pairs without creating large matrices.
            # We'll support single vector vs matrix or matrix vs matrix cautiously.
            if np.ndim(x1) == 1: x1 = x1[np.newaxis, :]
            if np.ndim(x2) == 1: x2 = x2[np.newaxis, :]
            
            # Distance matrix calculation
            x1_norm = np.sum(x1 ** 2, axis=1).reshape(-1, 1)
            x2_norm = np.sum(x2 ** 2, axis=1).reshape(1, -1)
            dist_sq = x1_norm + x2_norm - 2 * np.dot(x1, x2.T)
            return np.exp(-self.gamma * dist_sq)
        elif self.kernel_type == 'sigmoid':
            return np.tanh(np.dot(x1, x2.T) + 1)
        else:
            raise ValueError(f"Unknown kernel: {self.kernel_type}")

    def fit(self, X, y):
        """
        Train SVM using Simplified SMO algorithm.
        """
        n_samples, n_features = X.shape
        self.X = X
        self.y = y
        self.alpha = np.zeros(n_samples)
        self.b = 0
        
        # Precompute kernel matrix? If dataset is small (<5000), yes.
        # User dataset might be larger, so maybe cache or compute on fly. 
        # Given "new_train.csv" size isn't confirmed but likely < 10k based on context, we try full Kernel matrix.
        # If OOM, we switch to on-the-fly.
        
        K = self._kernel(X, X)
        
        iter_count = 0
        passes = 0
        
        # Simplified SMO (Platt 1998)
        while passes < self.max_iter:
            num_changed_alphas = 0
            for i in range(n_samples):
                # Calculate Error Ei
                # f(xi) = sum(alpha_j * y_j * K(xj, xi)) + b
                # But here we can use the precomputed Kernel row K[i]
                # Prediction for xi:
                f_xi = np.dot((self.alpha * self.y), K[:, i]) + self.b
                Ei = f_xi - y[i]
                
                if (y[i] * Ei < -self.tol and self.alpha[i] < self.C) or \
                   (y[i] * Ei > self.tol and self.alpha[i] > 0):
                    
                    # Randomly select j != i
                    j = i
                    while j == i:
                        j = np.random.randint(0, n_samples)
                        
                    f_xj = np.dot((self.alpha * self.y), K[:, j]) + self.b
                    Ej = f_xj - y[j]
                    
                    # Save old alphas
                    alpha_i_old = self.alpha[i]
                    alpha_j_old = self.alpha[j]
                    
                    # Compute L and H
                    if y[i] != y[j]:
                        L = max(0, self.alpha[j] - self.alpha[i])
                        H = min(self.C, self.C + self.alpha[j] - self.alpha[i])
                    else:
                        L = max(0, self.alpha[i] + self.alpha[j] - self.C)
                        H = min(self.C, self.alpha[i] + self.alpha[j])
                        
                    if L == H:
                        continue
                        
                    # Compute eta
                    eta = 2 * K[i, j] - K[i, i] - K[j, j]
                    
                    if eta >= 0:
                        continue
                        
                    # Update alpha_j
                    self.alpha[j] = alpha_j_old - (y[j] * (Ei - Ej)) / eta
                    
                    # Clip alpha_j
                    if self.alpha[j] > H:
                        self.alpha[j] = H
                    elif self.alpha[j] < L:
                        self.alpha[j] = L
                        
                    if abs(self.alpha[j] - alpha_j_old) < 1e-5:
                        continue
                        
                    # Update alpha_i
                    self.alpha[i] = alpha_i_old + y[i] * y[j] * (alpha_j_old - self.alpha[j])
                    
                    # Update b
                    b1 = self.b - Ei - y[i] * (self.alpha[i] - alpha_i_old) * K[i, i] - \
                         y[j] * (self.alpha[j] - alpha_j_old) * K[i, j]
                    b2 = self.b - Ej - y[i] * (self.alpha[i] - alpha_i_old) * K[i, j] - \
                         y[j] * (self.alpha[j] - alpha_j_old) * K[j, j]
                         
                    if 0 < self.alpha[i] < self.C:
                        self.b = b1
                    elif 0 < self.alpha[j] < self.C:
                        self.b = b2
                    else:
                        self.b = (b1 + b2) / 2
                        
                    num_changed_alphas += 1
            
            if num_changed_alphas == 0:
                passes += 1
            else:
                passes = 0
            
            # Should break if convergence is slow to avoid infinite loops in a simple script
            if iter_count > self.max_iter * 5: # Safety break
                 break
            iter_count += 1

        # Store only support vectors
        sv_indices = self.alpha > 1e-5
        self.support_vectors = X[sv_indices]
        self.support_vector_labels = y[sv_indices]
        self.support_vector_alphas = self.alpha[sv_indices]
        print(f"Training completed. Support vectors: {np.sum(sv_indices)}")

    def predict(self, X):
        if self.support_vectors is None:
            raise Exception("Model not trained yet")
            
        # Decision function: sum(alpha_i * y_i * K(x_i, x)) + b
        # We compute K between support vectors and Input X
        # K_matrix shape: (n_sv, n_test_samples)
        
        K_matrix = self._kernel(self.support_vectors, X)
        
        # (alphas * y) is shape (n_sv, )
        # We need dot product: (1, n_sv) dot (n_sv, n_test) -> (1, n_test)
        
        weight_vec = self.support_vector_alphas * self.support_vector_labels
        
        # Careful with matrix multiplication shapes
        # Linear/Poly kernel returns (n_sv, n_test) if X is (n_test, features) and we did dot(x1, x2.T)
        # where x1=SV (n_sv, f), x2=X (n_test, f)
        
        preds = np.dot(weight_vec, K_matrix) + self.b
        return np.sign(preds)

## Experiments and Results
We load the data, preprocess it, and run the experiments for Linear, Polynomial, and RBF kernels.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

# --- Configuration ---
TRAIN_FILE = r"c:\Users\mshar\Music\new_train.csv"
SAMPLE_SIZE = 400  # Subsample for training to speed up SMO
TEST_SIZE = 200    # Subsample for testing
RANDOM_STATE = 42

def preprocess_data(df):
    # Target
    df['y'] = df['y'].map({'yes': 1, 'no': -1})
    
    # Identify Categorical and Numerical
    # Based on inspection:
    categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 
                        'loan', 'contact', 'month', 'day_of_week', 'poutcome']
    numerical_cols = ['age', 'duration', 'campaign', 'pdays', 'previous']
    
    # One-Hot Encoding
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    
    # Normalize Numerical
    for col in numerical_cols:
        if col in df.columns:
            mean = df[col].mean()
            std = df[col].std()
            if std > 0:
                df[col] = (df[col] - mean) / std
    
    # Ensure all data is float
    df = df.astype(float)
    
    return df

def get_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == -1) & (y_pred == -1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return accuracy, precision, recall, f1, [[tn, fp], [fn, tp]]

def train_test_split_custom(X, y, test_size=0.2, random_state=None):
    if random_state:
        np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    test_samples = int(len(X) * test_size)
    
    test_idx = indices[:test_samples]
    train_idx = indices[test_samples:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def run():
    print("Loading Data...")
    df = pd.read_csv(TRAIN_FILE)
    
    # Shuffle and Subsample first to avoid processing everything if huge
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    
    # Preprocess
    print("Preprocessing...")
    df = preprocess_data(df)
    
    # Split X, y
    y = df['y'].values
    X = df.drop('y', axis=1).values
    
    # Subsample for SVM speed
    X_train_full, X_test_full, y_train_full, y_test_full = train_test_split_custom(X, y, test_size=0.2, random_state=RANDOM_STATE)
    
    X_train = X_train_full[:SAMPLE_SIZE]
    y_train = y_train_full[:SAMPLE_SIZE]
    X_test = X_test_full[:TEST_SIZE]  # Keep test set reasonable size
    y_test = y_test_full[:TEST_SIZE]
    
    print(f"Training Set: {X_train.shape}, Test Set: {X_test.shape}")
    
    results = []
    
    # --- Task 1: Linear SVM Tuning ---
    print("\\n--- Linear SVM Tuning ---")
    C_values = [0.01, 0.1, 1, 10, 100]
    best_linear_f1 = -1
    best_linear_model = None
    
    for C in C_values:
        print(f"Training Linear SVM (C={C})...")
        model = SVM(kernel='linear', C=C, max_iter=20) # Low max_iter for speed in testing
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc, prec, rec, f1, cm = get_metrics(y_test, y_pred)
        print(f"  Result: Acc={acc:.3f}, F1={f1:.3f}")
        results.append({'model': 'Linear', 'params': f'C={C}', 'accuracy': acc, 'f1': f1})
        
        if f1 > best_linear_f1:
            best_linear_f1 = f1
            best_linear_model = model

    # --- Task 2: Polynomial SVM Tuning ---
    print("\\n--- Polynomial SVM Tuning ---")
    degrees = [2, 3, 4, 5]
    poly_f1_scores = []
    best_poly_f1 = -1
    best_poly_model = None
    
    for d in degrees:
        print(f"Training Poly SVM (Degree={d})...")
        model = SVM(kernel='poly', degree=d, C=1.0, max_iter=20)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc, prec, rec, f1, cm = get_metrics(y_test, y_pred)
        poly_f1_scores.append(f1)
        print(f"  Result: Acc={acc:.3f}, F1={f1:.3f}")
        results.append({'model': 'Poly', 'params': f'Deg={d}', 'accuracy': acc, 'f1': f1})

        if f1 > best_poly_f1:
            best_poly_f1 = f1
            best_poly_model = model
            
    # --- Task 3: RBF SVM Tuning ---
    print("\\n--- RBF SVM Tuning ---")
    C_rbf = [0.1, 1, 10, 100]
    Gammas = [0.001, 0.01, 0.1, 1]
    best_rbf_f1 = -1
    best_rbf_model = None
    
    for C in C_rbf:
        for G in Gammas:
            print(f"Training RBF SVM (C={C}, Gamma={G})...")
            model = SVM(kernel='rbf', C=C, gamma=G, max_iter=20)
            try:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                acc, prec, rec, f1, cm = get_metrics(y_test, y_pred)
                print(f"  Result: Acc={acc:.3f}, F1={f1:.3f}")
                results.append({'model': 'RBF', 'params': f'C={C}, G={G}', 'accuracy': acc, 'f1': f1})
                
                if f1 > best_rbf_f1:
                    best_rbf_f1 = f1
                    best_rbf_model = model
            except Exception as e:
                print(f"  Failed: {e}")

    # --- Task 4: Ensemble ---
    print("\\n--- Ensemble ---")
    if best_linear_model and best_poly_model and best_rbf_model:
        print("Evaluating Ensemble of Top 3 Models...")
        pred1 = best_linear_model.predict(X_test)
        pred2 = best_poly_model.predict(X_test)
        pred3 = best_rbf_model.predict(X_test)
        
        # Majority Vote
        final_pred = np.sign(pred1 + pred2 + pred3)
        acc, prec, rec, f1, cm = get_metrics(y_test, final_pred)
        print(f"Ensemble Result: Acc={acc:.3f}, F1={f1:.3f}")
        results.append({'model': 'Ensemble', 'params': 'Top3', 'accuracy': acc, 'f1': f1})
        
        # Plot Confusion Matrix for Ensemble
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title('Ensemble Confusion Matrix')
        plt.ylabel('True')
        plt.xlabel('Predicted')
        plt.savefig('confusion_matrix_ensemble.png')
    
    # --- Plotting Poly Degree vs F1 ---
    plt.figure()
    plt.plot(degrees, poly_f1_scores, marker='o')
    plt.title('Polynomial Degree vs F1 Score')
    plt.xlabel('Degree')
    plt.ylabel('F1 Score')
    plt.grid(True)
    plt.savefig('poly_f1_plot.png')
    
    # --- Save Results ---
    res_df = pd.DataFrame(results)
    res_df.to_csv('svm_experiment_results.csv', index=False)
    print("\\nExperiments Completed. Results saved.")

if __name__ == "__main__":
    run()

Loading Data...
Preprocessing...
Training Set: (400, 48), Test Set: (200, 48)

--- Linear SVM Tuning ---
Training Linear SVM (C=0.01)...
Training completed. Support vectors: 99
  Result: Acc=0.940, F1=0.455
Training Linear SVM (C=0.1)...
Training completed. Support vectors: 96
  Result: Acc=0.960, F1=0.714
Training Linear SVM (C=1)...
Training completed. Support vectors: 93
  Result: Acc=0.940, F1=0.625
Training Linear SVM (C=10)...
Training completed. Support vectors: 135
  Result: Acc=0.925, F1=0.516
Training Linear SVM (C=100)...
Training completed. Support vectors: 136
  Result: Acc=0.890, F1=0.500

--- Polynomial SVM Tuning ---
Training Poly SVM (Degree=2)...
Training completed. Support vectors: 112
  Result: Acc=0.940, F1=0.667
Training Poly SVM (Degree=3)...
Training completed. Support vectors: 135
  Result: Acc=0.955, F1=0.727
Training Poly SVM (Degree=4)...
Training completed. Support vectors: 129
  Result: Acc=0.940, F1=0.600
Training Poly SVM (Degree=5)...
Training completed